# Payment fraud detection with Python (PaySim)

**Question:** Can a model find fraudulent payments better than the rules a payment company already has, and how many alerts would a fraud team need to review?

**The data:** PaySim is a public *simulated* dataset of mobile-money transactions (about 6.4 million rows). One row is one transaction. It is synthetic, so no real customers are in it. Columns:

| Column | Meaning |
|---|---|
| `step` | Time. One step is one hour of the simulation (743 steps, about 30 days) |
| `type` | `CASH_IN`, `CASH_OUT`, `DEBIT`, `PAYMENT` or `TRANSFER` |
| `amount` | Amount of the transaction |
| `oldbalanceOrg`, `newbalanceOrig` | Sender balance before and after |
| `oldbalanceDest`, `newbalanceDest` | Receiver balance before and after |
| `nameOrig`, `nameDest` | Account names. Merchant accounts start with `M` |
| `isFraud` | 1 if the transaction is fraud (the label I want to predict) |
| `isFlaggedFraud` | 1 if the simulated system's own rule flagged it |

**Plan:**
1. Load and check the data. Look at where the fraud is.
2. Build features and split the data **by time**, not at random
3. Compare simple and stronger models, and handle the very low share of fraud
4. Test the best model once on the last part of the period
5. Translate the result into review workload for a fraud team
6. Save tables for a dashboard

## 1. Set-up and data

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

DATA_DIR = Path("data")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

csv_files = sorted(DATA_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError("Put the PaySim CSV file in the data/ folder. The README has the source.")
DATA_PATH = csv_files[0]

# Set MAX_ROWS to a number (for example 1_000_000) to try the notebook quickly on a small part.
# The file is ordered by time, so this keeps only the first part of the period.
MAX_ROWS = None

In [ ]:
# Read in chunks to save memory. I keep only the columns I need, and turn the receiver name into one flag.
use_cols = ["step", "type", "amount", "oldbalanceOrg", "newbalanceOrig", "nameDest",
            "oldbalanceDest", "newbalanceDest", "isFraud", "isFlaggedFraud"]
parts = []
for chunk in pd.read_csv(DATA_PATH, usecols=use_cols, chunksize=1_000_000, nrows=MAX_ROWS):
    chunk["dest_is_merchant"] = chunk["nameDest"].str.startswith("M").astype("int8")
    chunk = chunk.drop(columns="nameDest")
    chunk["step"] = chunk["step"].astype("int16")
    chunk["isFraud"] = chunk["isFraud"].astype("int8")
    chunk["isFlaggedFraud"] = chunk["isFlaggedFraud"].astype("int8")
    parts.append(chunk)
df = pd.concat(parts, ignore_index=True)
df["type"] = df["type"].astype("category")
del parts

print(f"File: {DATA_PATH.name}")
print(f"Rows: {len(df):,}   Steps: {df['step'].min()} to {df['step'].max()}")
df.head()

In [ ]:
# Chart style: one colour for normal transactions and one for fraud, the same in every chart.
BLUE, ORANGE = "#2a78d6", "#eb6834"      # BLUE = normal / main series, ORANGE = fraud
INK, INK_SOFT, SURFACE, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e4e3de"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_SOFT, "axes.titlecolor": INK,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "xtick.color": INK_SOFT, "ytick.color": INK_SOFT, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "font.size": 10, "legend.frameon": False,
})

### Data checks and class balance

In [ ]:
print("Missing values per column:")
print(df.isna().sum().to_string())
print()
n_fraud = int(df["isFraud"].sum())
print(f"Fraud transactions: {n_fraud:,} of {len(df):,}  ({df['isFraud'].mean():.3%})")
print(f"Total fraud amount: {df.loc[df['isFraud'] == 1, 'amount'].sum():,.0f}")
print(f"A model that always says 'no fraud' would be right {1 - df['isFraud'].mean():.2%} of the time, and would catch nothing.")

Fraud is **very rare**, so a model that says "no fraud" every time is almost always right and catches nothing. *Accuracy* is a useless measure here. I use **precision** (of the alerts, how many are real fraud), **recall** (of the real fraud, how much is caught) and **average precision** (the area under the precision-recall curve).

### Where is the fraud?

In [ ]:
by_type = (df.assign(fraud_amount=df["amount"] * df["isFraud"])
           .groupby("type", observed=True)
           .agg(transactions=("isFraud", "size"), fraud=("isFraud", "sum"), fraud_amount=("fraud_amount", "sum")))
by_type["fraud rate (%)"] = by_type["fraud"] / by_type["transactions"] * 100
by_type = by_type.sort_values("fraud rate (%)", ascending=False)
by_type.round({"fraud rate (%)": 3, "fraud_amount": 0})

In [ ]:
fraud_types = sorted(df.loc[df["isFraud"] == 1, "type"].astype(str).unique())
print("Transaction types that contain fraud:", fraud_types)

In [ ]:
# Chart 1: share of transactions that are fraud, by type.
fig, ax = plt.subplots(figsize=(7.5, 3.8))
rates = by_type["fraud rate (%)"]
ax.bar(rates.index.astype(str), rates.values, color=ORANGE, width=0.6)
for i, v in enumerate(rates.values):
    ax.text(i, v + rates.max() * 0.02, f"{v:.2f}%", ha="center", fontsize=10)
ax.set_ylabel("Fraud rate (% of transactions)")
ax.set_title("Fraud only happens in some transaction types")
ax.grid(axis="x", visible=False)
plt.show()

From here on I work only with the transaction types that contain fraud (listed above). Every other type has zero fraud in this data, so a model has nothing to learn from it, and these types can be handled by a simple rule.

## 2. Features and a time-based split

In [ ]:
m = df[df["type"].astype(str).isin(fraud_types)].copy()

# Simulated clock: hour of the day
m["hour"] = (m["step"] % 24).astype("int8")
m["is_transfer"] = (m["type"].astype(str) == "TRANSFER").astype("int8")

# Balance checks. In a normal transaction, the balances change by exactly the amount.
# A gap between the expected and the real balance is a warning sign.
m["error_orig"] = m["newbalanceOrig"] + m["amount"] - m["oldbalanceOrg"]
m["error_dest"] = m["oldbalanceDest"] + m["amount"] - m["newbalanceDest"]

# Account-emptying patterns that are typical for fraud
m["orig_emptied"] = ((m["newbalanceOrig"] == 0) & (m["oldbalanceOrg"] > 0)).astype("int8")
m["dest_was_empty"] = (m["oldbalanceDest"] == 0).astype("int8")

FEATURES = ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest",
            "error_orig", "error_dest", "orig_emptied", "dest_was_empty", "hour", "is_transfer"]
print(f"Rows used for modelling: {len(m):,}   Fraud: {int(m['isFraud'].sum()):,} ({m['isFraud'].mean():.2%})")

### Why I split by time
In real life, a fraud model is trained on the past and used on the future. A random split would mix hours from the same period into training and testing, and the score would look better than it should. I split by **step (time)**:
- **Train:** the first 60% of rows (by time) to fit the models
- **Validation:** the next 15% to compare models and to choose the alert threshold
- **Test:** the last 25%. I use it **once**, at the end, for the final result.

In [ ]:
q60, q75 = m["step"].quantile([0.60, 0.75]).astype(int)
train = m[m["step"] <= q60]
valid = m[(m["step"] > q60) & (m["step"] <= q75)]
test = m[m["step"] > q75]

split_table = pd.DataFrame({
    "steps": [f"{train['step'].min()} to {train['step'].max()}", f"{valid['step'].min()} to {valid['step'].max()}",
              f"{test['step'].min()} to {test['step'].max()}"],
    "rows": [len(train), len(valid), len(test)],
    "fraud": [int(train["isFraud"].sum()), int(valid["isFraud"].sum()), int(test["isFraud"].sum())],
}, index=["train", "validation", "test"])
split_table["fraud rate (%)"] = (split_table["fraud"] / split_table["rows"] * 100).round(3)
assert (split_table["fraud"] > 0).all(), "Every part needs some fraud. Use more rows."
split_table

## 3. Models
I compare these on the **validation** set, using average precision:
1. **Logistic regression** (a simple, readable baseline) with class weights, so the rare fraud counts more
2. **Gradient boosting** without class weights
3. **Gradient boosting** with class weights ("balanced")

I also compare against two **rules**:
- the simulated system's own flag (`isFlaggedFraud`)
- a one-line rule: "the sender's account is emptied completely"

In [ ]:
def signed_log(x):
    """Log scale for money columns that can also be negative."""
    return np.sign(x) * np.log1p(np.abs(x))


models = {
    "Logistic regression (balanced)": make_pipeline(
        FunctionTransformer(signed_log), StandardScaler(),
        LogisticRegression(class_weight="balanced", max_iter=300)),
    "Gradient boosting": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=0),
    "Gradient boosting (balanced)": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1,
                                                                   class_weight="balanced", random_state=0),
}

rows = []
for name, model in models.items():
    model.fit(train[FEATURES], train["isFraud"])
    p = model.predict_proba(valid[FEATURES])[:, 1]
    rows.append({"model": name,
                 "validation average precision": average_precision_score(valid["isFraud"], p),
                 "validation ROC AUC": roc_auc_score(valid["isFraud"], p)})
comparison = pd.DataFrame(rows).set_index("model")
comparison.round(4)

In [ ]:
best_name = comparison["validation average precision"].idxmax()
best_model = models[best_name]
print("Best model on validation:", best_name)

# Choose the alert threshold on the validation set: the score that gives the best F1 (balance of precision and recall).
p_valid = best_model.predict_proba(valid[FEATURES])[:, 1]
prec, rec, thr = precision_recall_curve(valid["isFraud"], p_valid)
f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
threshold = float(thr[np.argmax(f1)])
print(f"Alert threshold (best F1 on validation): {threshold:.4f}")

## 4. Final test
Now I use the test set **once**, with the best model and the threshold chosen on validation.

In [ ]:
test = test.copy()
test["score"] = best_model.predict_proba(test[FEATURES])[:, 1]
test["alert"] = (test["score"] >= threshold).astype("int8")
test["rule_empty_account"] = test["orig_emptied"]


def classification_row(name, y_true, y_pred, score=None):
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"approach": name, "alerts": tp + fp, "caught fraud": tp, "false alarms": fp, "missed fraud": fn,
            "precision": precision, "recall": recall, "F1": f1,
            "average precision": average_precision_score(y_true, score) if score is not None else np.nan}


y = test["isFraud"]
results = pd.DataFrame([
    classification_row(f"Model: {best_name}", y, test["alert"], test["score"]),
    classification_row("Rule: system flag (isFlaggedFraud)", y, test["isFlaggedFraud"]),
    classification_row("Rule: sender account emptied", y, test["rule_empty_account"]),
]).set_index("approach")
results.round({"precision": 4, "recall": 4, "F1": 4, "average precision": 4})

In [ ]:
# Chart 2: precision-recall curve of the model on the test set, with the two rules as points.
p_c, r_c, _ = precision_recall_curve(y, test["score"])
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.plot(r_c, p_c, color=BLUE, linewidth=2, label=f"Model ({best_name})")
for label, row in [("System flag", results.iloc[1]), ("Account-emptied rule", results.iloc[2])]:
    ax.scatter(row["recall"], row["precision"], s=70, color=ORANGE, zorder=3, edgecolor=SURFACE, linewidth=1.5)
    ax.annotate(label, (row["recall"], row["precision"]), textcoords="offset points", xytext=(8, 6), fontsize=9, color=INK_SOFT)
ax.set_xlabel("Recall (share of all fraud that is caught)")
ax.set_ylabel("Precision (share of alerts that are real fraud)")
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.set_title("Test set: the model against two rules")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16))
plt.show()

## 5. What this means for a fraud team
A fraud team cannot check every transaction. A more useful question is: *if the team can review only the top X% of transactions by score, how much fraud do they catch?*

In [ ]:
order = np.argsort(-test["score"].to_numpy(), kind="stable")
fraud_flags = y.to_numpy()[order]
amounts = test["amount"].to_numpy()[order]
fraud_amounts = amounts * fraud_flags
total_fraud, total_fraud_amount = fraud_flags.sum(), fraud_amounts.sum()

rows = []
for share in [0.0005, 0.001, 0.0025, 0.005, 0.01, 0.02]:
    k = max(1, int(len(test) * share))
    rows.append({
        "transactions reviewed": f"{share:.2%} ({k:,})",
        "fraud caught (count)": f"{fraud_flags[:k].sum() / total_fraud:.1%}",
        "fraud caught (amount)": f"{fraud_amounts[:k].sum() / total_fraud_amount:.1%}",
        "precision": f"{fraud_flags[:k].mean():.1%}",
    })
workload = pd.DataFrame(rows).set_index("transactions reviewed")
workload

In [ ]:
# Chart 3: which features matter? (permutation importance: how much does average precision drop when one feature is shuffled?)
sample = test.sample(n=min(200_000, len(test)), random_state=0)
imp = permutation_importance(best_model, sample[FEATURES], sample["isFraud"], scoring="average_precision",
                             n_repeats=3, random_state=0)
importance = pd.Series(imp.importances_mean, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.barh(importance.index, importance.values, color=BLUE, height=0.6)
ax.set_xlabel("Drop in average precision when the feature is shuffled")
ax.set_title("Which features does the model use?")
ax.grid(axis="y", visible=False)
plt.show()

**A word of caution about these results.** PaySim is **simulated**. Its fraud follows simple patterns that a model can find easily. Real payment fraud is messier, so scores on real data would be lower, and a real system would also need features such as customer history, device and location. This project shows the method: time-based validation, handling of rare fraud, honest comparison with rules, and translation into review workload. It does not show how well a model would work on real payments.

## 6. Result and tables for a dashboard

In [ ]:
top = results.iloc[0]
flag, rule = results.iloc[1], results.iloc[2]
k1 = max(1, int(len(test) * 0.005))
print(f"Best model: {best_name}")
print(f"Test set: {len(test):,} transactions of the fraud-prone types, {int(total_fraud):,} fraud cases.")
print(f"At the chosen threshold the model raised {int(top['alerts']):,} alerts: "
      f"precision {top['precision']:.1%}, recall {top['recall']:.1%} ({int(top['caught fraud']):,} caught, {int(top['missed fraud']):,} missed).")
print(f"The system's own flag caught {flag['recall']:.1%} of the fraud; the 'account emptied' rule caught {rule['recall']:.1%} with precision {rule['precision']:.1%}.")
print(f"Reviewing only the top 0.5% of transactions by score ({k1:,}) would catch {fraud_flags[:k1].sum() / total_fraud:.1%} of the fraud cases "
      f"and {fraud_amounts[:k1].sum() / total_fraud_amount:.1%} of the fraud amount.")

In [ ]:
# Tables for a dashboard (Looker Studio, Power BI). All are built from the data above.
fraud_by_type = by_type.reset_index().rename(columns={"fraud_amount": "fraud_amount_total"})
fraud_by_type["type"] = fraud_by_type["type"].astype(str)

hour_table = (m.groupby("hour").agg(transactions=("isFraud", "size"), fraud=("isFraud", "sum"))
              .assign(fraud_rate_pct=lambda d: d["fraud"] / d["transactions"] * 100).reset_index())

bands = pd.cut(m["amount"], [-0.01, 1_000, 10_000, 100_000, 1_000_000, np.inf],
               labels=["up to 1k", "1k to 10k", "10k to 100k", "100k to 1M", "over 1M"])
amount_table = (m.groupby(bands, observed=True).agg(transactions=("isFraud", "size"), fraud=("isFraud", "sum"))
                .assign(fraud_rate_pct=lambda d: d["fraud"] / d["transactions"] * 100).reset_index()
                .rename(columns={"amount": "amount_band"}))
amount_table["amount_band"] = amount_table["amount_band"].astype(str)

daily = (test.assign(day=(test["step"] - 1) // 24 + 1,
                     caught=test["alert"] * test["isFraud"],
                     fraud_amount=test["amount"] * test["isFraud"],
                     caught_amount=test["amount"] * test["isFraud"] * test["alert"])
         .groupby("day")
         .agg(transactions=("isFraud", "size"), fraud=("isFraud", "sum"), alerts=("alert", "sum"),
              caught_fraud=("caught", "sum"), fraud_amount=("fraud_amount", "sum"),
              caught_fraud_amount=("caught_amount", "sum"))
         .reset_index())

fraud_by_type.round(4).to_csv(OUT_DIR / "fraud_by_type.csv", index=False)
hour_table.round(4).to_csv(OUT_DIR / "fraud_by_hour.csv", index=False)
amount_table.round(4).to_csv(OUT_DIR / "fraud_by_amount_band.csv", index=False)
daily.round(2).to_csv(OUT_DIR / "test_daily_monitoring.csv", index=False)
workload.reset_index().to_csv(OUT_DIR / "review_workload.csv", index=False)
results.reset_index().round(4).to_csv(OUT_DIR / "approach_comparison.csv", index=False)
print("Saved to outputs/:", sorted(p.name for p in OUT_DIR.glob("*.csv")))